In [17]:
from typing import List, TypedDict
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from litellm import responses_with_retries
from litellm.llms.vllm.completion.handler import llm
from typer.cli import state
import os

In [18]:
load_dotenv()

API_KEY = os.getenv("API_KEY")
MODEL = os.getenv("MODEL")
BASE_URL = os.getenv("BASE_URL")


In [19]:
class AgentState(TypedDict):
    message: List[HumanMessage]

llm = ChatOpenAI(model=MODEL, api_key=API_KEY, base_url=BASE_URL)

def process(state: AgentState) -> AgentState:
    response = llm.invoke(state["message"])
    print(f"\nAI: {response.content}")
    return state

In [20]:
graph = StateGraph(AgentState)

PROCESS = "process"

graph.add_node(PROCESS, process)
graph.add_edge(START, PROCESS)
graph.add_edge(PROCESS, END)

agent = graph.compile()

In [21]:
user_input = input("Enter: ")
while user_input != "exit":
    agent.invoke({"message": [HumanMessage(content=user_input)]})
    user_input = input("Enter : ")


AI: Hello! How can I help you today?
